**Processed by: Guy Kahana & Anat Peled**

This project is the 3rd assignment required in Data Science Course @Naya College and submitted to the instructor Dror Geva.
The project main purpose was to explore and practice the ML regression processes

In [ ]:
!pip install pydotplus
!pip install six

In [ ]:
# General tools
import os
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from IPython.display import Image, HTML
from plotnine import *
import pydot
from plotnine import *
from tqdm import tqdm

# For transformations and predictions
from scipy.optimize import curve_fit
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import pairwise_distances
from sklearn.tree import DecisionTreeRegressor, export_graphviz
from six import StringIO

# For scoring
from sklearn.metrics import mean_squared_error as mse

# For validation
from sklearn.model_selection import train_test_split as split

# sns.set_theme(style="darkgrid")

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**About The Data**

1. The dataset was uploaded to Kaggle in 06/2020 by *Yamaç Eren Ay*, who used the Spotify Web API for develepors to build a data that contains more than 160,000 songs. These are NOT all the spotify tracks, but quite alot of it...

2. The task in this dataset is to predict the popularity of the track according to the track's features. 

3. Each row in the data represent a unique track, identified by a unique ID feature generated by Spotify.

4. The Features of the tracks are: 

# **A First Glance...**

In [ ]:
path = '/kaggle/input/spotify-dataset-19212020-160k-tracks/data.csv'
df = pd.read_csv(path)
df.head()

In [ ]:
df.info()

# **Cleaning**

## **Features Selection**

Looking closely at the features, there are some features to consider whether to include in the final model or not:

* **id:** id is unique for each track, therfore cannot assist a model and will be dropped.
* **name:** There are 132,940 unique values. In our opinion, this is a problematic categorical feature to insert in a model, and will be dropped.
* **artists:** there are 33,375 unique features. Since we enticipate the artist will influence the popularity of the track, this feature will remain and be handled seperatley.
* **release_date \ year:** The year feature is a derivative to the release_date feature, therfore one of them should be excluded. Since the release date contains rows with full date and others with year only, the release_date will be dropped.  

In [ ]:
columns = ['id', 'name', 'artists', 'release_date', 'year']
for col in columns:
  print(f'{col:<15}: {df[col].nunique()} unique values')

In [ ]:
df = df.drop(labels=['id', 'name', 'release_date'], axis=1)
df.shape

## **Duplicates & Nulls**

Prior to the feature dropped above, there were not any duplicates in the data set, due to the unique id feature.

After the a.m. features removal, there are 669 duplicates in the dataset.

Only the first copy of the duplicate will remain for the model.

There are not null values in the data set. 

In [ ]:
df.duplicated().sum()

In [ ]:
df = df[~df.duplicated()==1]
df.shape

In [ ]:
df.isnull().sum().sum()

# **EDA**
---

## **Overview**

In [ ]:
numeric_columns = df.columns[df.dtypes != 'object']
string_columns = df.columns[df.dtypes == 'object']
print(f'There are {len(numeric_columns)} numeric columns & {len(string_columns)} string columns')

In [ ]:
# Numeric Heatmap
numeric_df = pd.DataFrame(data=df, columns=numeric_columns, index=df.index)
corr = np.abs(numeric_df.corr())
fig, ax = plt.subplots(figsize=(8, 8))
cmap = sns.color_palette("magma")
sns.heatmap(corr, cmap=cmap, square=True)
plt.title('Correlation between numerical features: abs values')
plt.show()

In [ ]:
series = np.abs(corr['popularity']).sort_values(ascending=False)
print('The most linear correlated features to POPULARITY are:')
for i, row in enumerate(series):
    if 0.2 <= row < 1:
      print(f'{series.index[i]:17} --> {row: .2f} (abs)')

In [ ]:
# 'Pairplot of Numerical Features exc. dummies'
# Data is sampled here due to long running time
sns_plot = sns.pairplot(df.sample(1000), height=1, vars=['popularity', 'acousticness', 'danceability', 'duration_ms', 'energy',
       'instrumentalness', 'key', 'liveness', 'loudness', 
       'speechiness', 'tempo', 'valence', 'year'])
        # sns_plot.savefig('pairplot.png')
plt.show()

## **Categorical Features**
---

### **Artists**

In [ ]:
df['artists'].nunique()

In [ ]:
# # Plotting
fig, ax = plt.subplots(figsize = (12, 10))
lead_artists = df.groupby('artists')['popularity'].sum().sort_values(ascending=False).head(20)
ax = sns.barplot(x=lead_artists.values, y=lead_artists.index, palette="Greens", orient="h", edgecolor='black', ax=ax)
ax.set_xlabel('Sum of Popularity', c='r', fontsize=12)
ax.set_ylabel('Artist', c='r', fontsize=12)
ax.set_title('20 Most Popular Artists in Dataset', c='r', fontsize=14, weight = 'bold')
plt.show()

In [ ]:
# find out artists with more than 100 tracks
df['popular_artist'] = df['artists'].map(df['artists'].value_counts()>100)
pop_arts  = df.groupby(['artists', 'popular_artist'])['popularity'].mean().sort_values(ascending=False).reset_index(1)
df_pop_arts = pop_arts.loc[pop_arts['popular_artist'] == True,['popularity']]

# # Plotting
fig, ax = plt.subplots(figsize = (12, 10))
lead_artists = df_pop_arts.groupby('artists')['popularity'].mean().sort_values(ascending=False).head(20)
ax = sns.barplot(x=lead_artists.values, y=lead_artists.index, palette="Greens", orient="h", edgecolor='black', ax=ax)
ax.set_xlabel('Mean of Popularity', c='r', fontsize=12)
ax.set_ylabel('Artist', c='r', fontsize=12)
ax.set_title('20 Most Popular Artists in Dataset with > 100 tracks', c='r', fontsize=14, weight = 'bold')
plt.show()

There are 33,375 different artists in this feature.

Since we believe there should be a correlation between the artist and the popularity of the track, we would like to use the **Target Encoding** approach, and replace the artist with some derivative of its popularity.

First, we check distribution of number of times artists appear in the data.

It is clear that most artists apear less than 50 times

In [ ]:
# Adding Mean & Count values to each artist
df['mean'] = df.groupby('artists')['popularity'].transform('mean')
df['count'] = df.groupby('artists')['popularity'].transform('count')
# plotting
fig, ax = plt.subplots(figsize = (20, 3))
ax = sns.distplot(df['count'], bins = 600)
ax.set_xlabel('Count of apperances in data', fontsize=12, c='r')
ax.set_ylabel('% of artists', fontsize=12, c='r')
plt.show()

Lets zoom in to the left of the scale, to decide what is the cut-off for using artist mean OR overall data mean:

In [ ]:
fig, ax = plt.subplots(figsize = (15, 3))
ax = sns.distplot(df['count'], bins=600, kde=False)
ax.set_xlabel('Count of appearances in data', fontsize=12, c='r')
ax.set_ylabel('# of artists', fontsize=12, c='r')
ax.set_xlim(1,20)
ax.set_xticks(range(1,21,1))
ax.axvline(x=3, ymin=0, ymax=1, color='orange', linestyle='dashed', linewidth=3)
font = {'family': 'serif',
        'color':  'red',
        'weight': 'bold',
        'size': 14,
        }
ax.annotate("", xy=(3, 19000), xytext=(4.8, 19000), arrowprops=dict(arrowstyle="->", color='r', linestyle='dashed', linewidth=3))
ax.text(x = 5, y = 19000, s='cutoff = 3', fontdict=font)

plt.show()

We also check for correlation between the number of appearances in data and the mean of popularity.

It is clear that artists that appear in the data more than 600 times, are getting almost 0 popularity.

there are 4 artists [Francisco Canaro, Ignacio Corsini, Эрнест Хемингуэй, Эрих Мария Ремарк] responsible for 3,546 tracks!

In [ ]:
fig, ax = plt.subplots(figsize = (15, 3))
stat = df.groupby('count')['mean'].mean().to_frame().reset_index()
ax = stat.plot(x='count', y='mean', marker='.', linestyle = '', ax=ax)
ax.set_xlabel('Count of appearances in data', fontsize=12, c='r')
ax.set_ylabel('Mean Popularity', fontsize=12, c='r')
plt.show()

In [ ]:
frequent_flyer = df[['artists','mean']].loc[df['count']>600].value_counts(ascending=False).reset_index()
frequent_flyer.columns=['Artist','Mean Popularity', '# of Tracks']
frequent_flyer

Therefore to handle this feature we will use a transformer that transforms the 'artists' feature from object
       to a float value as follows:
       
1. Replace the data with the artists mean popularity

2. Replace values where artists appear less than minimal times with popularity mean

3. Replace values where artists appear more than maximaum times with 0

### **Explicit**

Explicit content usually gets The Parental Advisory label (abbreviated PAL) which is a warning label. 

It is placed on audio recordings in recognition of profanity or inappropriate references, with the intention of alerting parents of material potentially unsuitable for children. 

The label has been included on digital listings offered by online music stores (as Spotify).

* Linear correlation to popularity is 0.2134

In [ ]:
fig, ax = plt.subplots(figsize = (15, 5))
y_ticks = ['No Explicit\nContent', 'Explicit\nContent']
ax = sns.barplot(x = (df['explicit'].value_counts(normalize=True)*100),
                 y= y_ticks, 
                 data=df, palette='Spectral')
ax.set_xlabel('% of records', fontsize=12, weight='bold')
ax.set_xlim(0.0,100.0)
ax.set_xticks(ticks=range(0,101,10))
ax.set_yticklabels(labels=y_ticks, fontsize=12)
ax.text(s='91.48%\n(154,829)', x=1, y=0.1, fontdict={'weight': 'bold', 'c':'red'})
ax.text(s='8.51%\n(14,411)', x=1, y=1.1, fontdict={'weight': 'bold', 'c':'red'})

plt.suptitle('Occurence of Explicit Data', fontsize=14, c='r')
plt.show()

In [ ]:
exp_data = sns.catplot(data=df, kind="bar",
                        x="explicit", y="popularity",
                        ci="sd", palette='Accent',
                        height=6)
exp_data.fig.suptitle('Popularity vs. Explicit', y=1.05, c='r')
exp_data.set_axis_labels("", "Popularity")
exp_data.set_xticklabels(['No Explicit\nContent', 'Explicit\nContent'])
plt.show()

### **Key**

* The 11-values of keys distribute between 4-13% each 

* Keys popularity' std is aprox. 20, makes it difficult to establish any correlation.

* Key 3 has the lowest mean (of 24.67) 

In [ ]:
df['duration_ms'].corr(df['popularity'])

In [ ]:
fig, ax = plt.subplots(figsize = (15, 5))
key_data = df['key'].value_counts(normalize=True, sort=True, ascending=True)*100
y_ticks = df['key'].value_counts().index
ax = sns.barplot(x = key_data.values, y= y_ticks, data = key_data, orient='h', palette='Spectral')
ax.set_xlabel('% of records', fontsize=12, weight='bold')
ax.set_ylabel('Key', fontsize=12, weight='bold')
ax.set_xlim(0.0,15.0, 0)
ax.set_xticks(ticks=range(0,16,1))
ax.set_yticklabels(labels=y_ticks, fontsize=12) #

rects = ax.patches

# For each bar: Place a label
for rect in rects:
    # Get X and Y placement of label from rect.
    x_value = rect.get_width() #f'{rect.get_width():.2f}'
    y_value = rect.get_y() + rect.get_height() / 2
    # Number of points between bar and label. Change to your liking.
    space = 5
    # Vertical alignment for positive values
    ha = 'left'

    # Use X value as label and format number with one decimal place
    label = f'{x_value:.1f}'

    # Create annotation
    plt.annotate(
        label,                      # Use `label` as label
        (x_value, y_value),         # Place label at end of the bar
        xytext=(space, 0),          # Horizontally shift label by `space`
        textcoords="offset points", # Interpret `xytext` as offset in points
        va='center',                # Vertically center label
        ha=ha)                      # Horizontally align label differently for
                                    # positive and negative values.

plt.suptitle('Distribution of Key (%)', fontsize=14, c='r')
plt.show()

In [ ]:
exp_data = sns.catplot(data=df, kind="bar",
                        x="key", y="popularity",
                        palette='pastel',
                        ci = 'sd',
                        height=5, aspect = 3)
exp_data.fig.suptitle('Popularity by Key Category', y=1.05, c='r', weight='bold')
exp_data.set_axis_labels("key", "Popularity")
plt.show()

### **Mode**

* Mode is a binary feature.

* There is no distinguish linear connection between mode and popularity (corr = -0.033)

In [ ]:
ax = df.groupby('mode')['popularity'].count().plot.pie(labels=['Minor', 'Major'],
                                 autopct='%1.1f%%', 
                                 fontsize=12)
ax.set_ylabel('')
ax.set_title('Mode Ratios')
ax.legend(labels=['Minor (n = 49298)', 'Major (n = 119942)'], bbox_to_anchor=(1, 1))
a = ax.get_anchor
plt.show()

In [ ]:
exp_data = sns.catplot(data=df, kind="bar",
                        x="mode", y="popularity",
                        ci="sd", palette='Accent',
                        height=4)
exp_data.fig.suptitle('Popularity vs. Mode', y=1.05, c='r')
exp_data.set_axis_labels("", "Popularity")
exp_data.set_xticklabels(['Minor (0)', 'Major(1)'])
plt.show()

## **Numeric Features**
---

### **Acousticness**

> The acousticness of the majority of tracks is either close to 0 or 1

> The more confidence we have that a track is acoustic, the less is its mean popularity (apply for acousitcness > 0.1)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
sns.distplot(df['acousticness'], kde=False, bins=30)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
ax1_data =  df.groupby('acousticness')['popularity'].mean().to_frame().reset_index()
ax = sns.scatterplot(x = ax1_data['acousticness'], y = ax1_data['popularity'], color='blue', ax=ax)
ax.set_title('Acousticness vs. Mean Popularity')
ax.set_ylabel('Mean Popularity', fontsize=12)
plt.tight_layout()
plt.show()

### **Danceability**

>Danceability- measure if the song is dancable based on the combination of tempo rythm and beat strength

>Danceability seems to have a normal distribution between 0-1

>the majoirity of Danceability values is between 0.2-0.8 with mean 0.53

In [ ]:
a= df['danceability'].corr(df['popularity']).round(2)
print(f'The pearדon corr is -->{a:^10}')

In [ ]:
df['danceability'].describe()

In [ ]:
sns.distplot(df['danceability'],kde=False, bins=30)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, figsize=(15, 6), sharey=True, sharex = True)
ax_data =  df.groupby('danceability')['popularity'].mean().to_frame().reset_index()
ax = sns.scatterplot(x='danceability', y='popularity', data=ax_data, color='blue', ax=ax)
ax.set_title('danceability')
ax.set_ylabel('Mean Popularity', fontsize=12)
plt.tight_layout()
plt.show()

### **Duration_ms**

* Conveting the miliseconds to minutes, so the column is more intuitive

* mean (3.85 min) and median (3.47 min) are close

* Some tracks are unexpectedly long... to understand the data it was divided to long tracks and short tracks:

    * Long tracks are mainly classic music and atmosphere music

    * plottibg it vs. mean popularity, it seems as a natural extension

    * There are not enoughe samples of tracks longer than 45 minutes

In [ ]:
# convert the miliseconds to minutes
df['duration_min'] = df['duration_ms']/60000
df['duration_min'].describe()

In [ ]:
long_tracks = df.loc[df['duration_min']>20]
short_tracks = df.loc[df['duration_min']<=20]

In [ ]:
fig, ax = plt.subplots(figsize = (15, 4))
ax = sns.distplot(short_tracks['duration_min'], kde = False)
ax.set_title(f'Short Tracks (<=20 min): {short_tracks.shape[0]} tracks')
ax.set_xticks(range(0,20,1))
ax.set_xlim(0,20)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (15, 5))
ax = sns.distplot(long_tracks['duration_min'], kde=False, bins=60)
ax.set_title(f'Long Tracks (>20 min): {long_tracks.shape[0]} tracks')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

# Short Tracks
ax1_data =  short_tracks.groupby('duration_min')['popularity'].mean().to_frame().reset_index()
ax1 = sns.scatterplot(x='duration_min', y='popularity', data=ax1_data, color='blue', ax=ax1)
ax1.set_xticks(range(0,20,1))
ax1.set_xlim(0,20)
ax1.set_title('Short tracks')

# Long Tracks
ax2_data =  long_tracks.groupby('duration_min')['popularity'].mean().to_frame().reset_index()
ax2 = sns.scatterplot(x=ax2_data['duration_min'], y=ax2_data['popularity'], color='green', ax=ax2)
ax2.set_xticks(range(20,101,10))
ax2.set_xlim(20,100)
ax2.set_title('Long Tracks')
font = {'family': 'serif',
        'color':  'red',
        'weight': 'normal',
        'size': 16,
        }
ax1.set_ylabel('Mean Popularity', fontsize=12)
plt.tight_layout()
plt.show()

* There is a small sampled data for tracks longer than 45 min. 

* Transformer \ pre-processing: sklearn.preprocessing.MinMaxScaler()

### **Energy**

>mesure the intensity and activity, energetic track feels faster, louder and noisier.

>It seem tat there is a releativly strong linear correlation of 0.497 correlation to the target.

In [ ]:
a= df['energy'].corr(df['popularity']).round(3)
print(f'The pearson corr is -->{a:^10}')

In [ ]:
df['energy'].describe()

In [ ]:
sns.distplot(df['energy'], kde = False, bins=30)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, figsize=(15, 6), sharey=True, sharex = True)
ax_data =  df.groupby('energy')['popularity'].mean().to_frame().reset_index()
ax = sns.scatterplot(x='energy', y='popularity', data=ax_data, color='blue', ax=ax)
ax.set_title('energy')
ax.set_ylabel('Mean Popularity', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
df_test=df[['energy','popularity']]
ggplot(df_test, aes(x='energy', y='popularity')) + geom_bin2d(bins = 20)

### **Instrumentalness**

> The closer the Instrumentalness to 1 - the greater liklyhood the track contains NO vocals

> Most of the data is distributed in both sides of scope.

> We decide to divide the feature into three categorical groups and create a onehotencoder transformer to reshpae our data

In [ ]:
a= df['instrumentalness'].corr(df['popularity']).round(3)
print(f'The pearson corr is -->{a:^10}')

In [ ]:
fig, ax = plt.subplots(figsize = (15,3))
ax = sns.distplot(df['instrumentalness'], kde = False, bins=30)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, figsize=(15, 6), sharey=True, sharex = True)
ax_data =  df.groupby('instrumentalness')['popularity'].mean().to_frame().reset_index()
ax = sns.scatterplot(x='instrumentalness', y='popularity', data=ax_data, color='blue', ax=ax)
ax.set_title('instrumentalness')
ax.set_ylabel('Mean Popularity', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
criteria= [df['instrumentalness'].between(0, 0.1),
           df['instrumentalness'].between(0.1000001, 0.95),
           df['instrumentalness'].between(0.950001, 1)]
values = [1, 2, 3]
df['instrumentalness_criteria'] = np.select(criteria, values, 0)
fig, ax = plt.subplots()
sns.distplot(df['instrumentalness_criteria'],ax=ax, kde=False, bins=3 )
ax.set_xticks(range(1,4))
plt.show()

### **liveness**

>Detects the presence of an audience. high livness suggests of the track was live

In [ ]:
a= df['liveness'].corr(df['popularity']).round(3)
print(f'The pearson corr is -->{a:^10}')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
sns.distplot(df['liveness'], kde = False, bins=30)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, figsize=(15, 6), sharey=True, sharex = True)
ax_data =  df.groupby('liveness')['popularity'].mean().to_frame().reset_index()
ax = sns.scatterplot(x='liveness', y='popularity', data=ax_data, color='blue', ax=ax)
ax.set_title('liveness')
ax.set_ylabel('Mean Popularity', fontsize=12)
plt.tight_layout()
plt.show()

### **Loudness**

In [ ]:
df['loudness'].corr(df['popularity'])

In [ ]:
df['loudness'].describe()

In [ ]:
fig, ax = plt.subplots(figsize = (15, 6))
sns.scatterplot(x='loudness', y='popularity', data=df, color='blue', alpha=0.3)
plt.show()

In [ ]:
# evaluate the amount of tracks in different ranges of loudness
for i in range(5,-60,-5):
  print(f"There are {df.loc[df['loudness'] < i, 'loudness'].count()} tracks below {i}")

### **Popularity (Target)**

In [ ]:
# Describe - all
df['popularity'].describe()

In [ ]:
# Describe - Popularity > 0
df.loc[df['popularity']>0,'popularity'].describe()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 11))
ax1 = sns.distplot(df['popularity'], kde=False, ax=ax1)
ax2 = sns.distplot(df.loc[df['popularity']>0,'popularity'], ax=ax2, kde=False)
ax1.set_xlim(0,100)
ax2.set_xlim(0,100)
ax1.set_xlabel('')
ax1.set_title('Up: All Data, Bottom: Popularity > 0', c='r', weight='bold')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
ax = df.groupby('year')['popularity'].mean().plot()
ax.set_title('Mean Popularity over the years', c='r', weight='bold')
ax.set_ylabel('Mean Popularity', weight='bold')
ax.set_xlabel('Year', weight='bold')
ax.set_xticks(range(1920, 2021, 5))
plt.show()

### **Speechiness**

>Due to an obvious change in in speechiness trand vs. poopularity, we decided to divide the records into two sub-groups.
The cutoff point is based on a decision tree model.

In [ ]:
a= df['speechiness'].corr(df['popularity']).round(2)
print(f'The pearson corr is -->{a:^10}')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
sns.distplot(df['speechiness'], kde = False, bins=30)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, figsize=(15, 6), sharey=True, sharex = True)
ax_data =  df.groupby('speechiness')['popularity'].mean().to_frame().reset_index()
ax = sns.scatterplot(x='speechiness', y='popularity', data=ax_data, color='blue', ax=ax)
ax.axvline(x=0.62,ymin=0,ymax=1,color='red',linestyle='dashed')
font = {'family': 'serif',
        'color':  'red',
        'weight': 'normal',
        'size': 16,
        }
ax.text(x = 0.63, y = 55, s='Split Value-0.62',fontdict=font)
ax.set_title('speechiness')
ax.set_ylabel('Mean Popularity', fontsize=12)
plt.tight_layout()
plt.show()

#### **Decision Tree**

In [ ]:
import pydotplus

X_sp = df['speechiness'].to_frame()
y_sp = df['popularity']
X_train, X_test, y_train, y_test = split(X_sp, y_sp , test_size = 0.2 , random_state = 42 )
model = DecisionTreeRegressor(max_leaf_nodes=2)
model.fit(X_train, y_train)

dot_data = StringIO()
export_graphviz(model, out_file=dot_data)
graph = pydotplus.graph_from_dot_data(dot_data.getvalue())
Image(graph.create_png())

In [ ]:
y_pred = model.predict(X_train)
rmse = np.sqrt(mse(y_train, y_pred))
print(f"RMSE_Train = {rmse:.2f}")

In [ ]:
y_test_pred = model.predict(X_test)
rmse = np.sqrt(mse(y_test, y_test_pred))
print(f"RMSE_test = {rmse:.2f}")

### **Tempo**

* The linear correlation between tempo and popularity is 0.1335.

* Outliers - there is a 145 values of 0. need to be replaced. 

In [ ]:
a = df['tempo'].corr(df['popularity']).round(2)
print(f"Linear correlation is {a}")

In [ ]:
sns.jointplot(x='tempo', y='popularity', data=df)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (15, 6))
ax = sns.distplot(df['tempo'], bins=250, kde=False)
ax.text(s='145\nOutliers?', x= 5, y=700, fontdict={'size': 12, 'c': 'darkred'})
ax.text(s='All values except 0', x= 125, y=2750, fontdict={'size': 12, 'c': 'darkred'})
ax.text(s='Corrected\nmedian\n114.87', x= 116, y=300, fontdict={'size': 10, 'c': 'darkgreen', 'weight': 'bold'})
ax.axvline(x=114.87, ymin=0, ymax=0.7, color='green', linestyle='dashed', linewidth=2)
ax.axvline(x=30.95, ymin=0, ymax=1, color='orange', linestyle='dashed', linewidth=3)
ax.axvline(x=244.09, ymin=0, ymax=1, color='orange', linestyle='dashed', linewidth=3)
ax.annotate("", xy=(31, 2700), xytext=(244, 2700), arrowprops=dict(arrowstyle="<->", color='r', linestyle='dashed', linewidth=2))
ax.annotate("", xy=(0, 300), xytext=(0, 900), arrowprops=dict(arrowstyle="->", color='r', linestyle='dashed', linewidth=3))
plt.show()

This feature will need an imputer and scaler.

### **Valence**

In [ ]:
a = df['valence'].corr(df['popularity']).round(2)
print(f"Linear correlation is {a}")

In [ ]:
sns.jointplot(x='valence', y='popularity', data=df)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (15, 6))
sns.scatterplot(x='valence', y='popularity', data=df, color='blue', alpha=0.3)
plt.show()

### **Year**

* The year attribute is linearly close to the popularity target (corr = 0.88)

* Each year contains different quantities of tracks. Most of the years are up to 2000, probably due to the 2000 maximal batch limit in the Spoyify's API.

* There are some values with 0 popularity in 2020. These is a relatively new songs in the dataset, from 6/2020, prior to data extraction.

In [ ]:
a = df['year'].corr(df['popularity']).round(2)
print(f"Linear correlation is {a}")

In [ ]:
fig, ax = plt.subplots(figsize=(17, 4))
ax = sns.distplot(df['year'], bins = 100, kde = False)
ax.set_xlim(1920,2020)
ax.set_xticks(range(1920, 2021, 10))
ax.set_title('# of tracks per Year')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax = df['year'].loc[df['popularity']==0].plot.hist(bins=50)
ax.set_title('0 Popularity by Year')
ax.text(s='Outliers?', x= 2015, y=1000, fontdict={'size': 12, 'c': 'darkred'})
ax.annotate("", xy=(2019, 300), xytext=(2019, 900), arrowprops=dict(arrowstyle="->", color='r', linestyle='dashed', linewidth=3))
plt.show()

**Due to the very high linear correlation between the year and the popularity, this feature will be excluded from our models.**

# **Pre-Processing**

First, we import a clean version of the data, without id, name, year and release_date and remove the duplicates.

In [ ]:
# Read column names from file
cols = list(pd.read_csv(path, nrows =1))
df = pd.read_csv(path, usecols=[i for i in cols if i not in ['id','name','release_date','year']])

# Remove duplicated
df = df[~df.duplicated()==1]
# df = df.sample(frac=0.3)
#Split the data to train and test
X_train, X_test, y_train, y_test = split(df.drop('popularity', axis=1), df['popularity'], test_size = 0.2, random_state = 12345)

After exploring the data, some features need pre-processing. This will be done in the following order:

1. Unique feature transformers

2. OneHotEncoder (aka dummies) for relevant features

3. MinMaxScaling for relevant features

4. Target scaling

### **AritistsTransformer**

Here we need to replace object data of the artists with some numerical indicator that identify the artist.

In [ ]:
class ArtistsTransformer():
  """ This transformer recives a DF with a feature 'artists' of dtype object
      and convert the feature to a float value as follows:
      1. Replace the data with the artists mean popularity
      2. Replace values where artists appear less than MinCnt with y.mean()
      3. Replace values where artists appear more than MaxCnt with 0
      
      PARAMETERS:
      ----------
      MinCnt (int): Minimal treshold of artisits apear in dataset, default = 3
      MaxCnt (int): Maximal treshold of artisits apear in dataset, default = 600

      RERTURN:
      ----------
      A DataFrame with converted artists str feature to ordinal floats
  """

  def __init__(self, MinCnt = 3.0, MaxCnt = 600.0):
      self.MinCnt = MinCnt
      self.MaxCnt = MaxCnt
      self.artists_df = None
  
  def fit (self, X, y):
      self.artists_df =  y.groupby(X.artists).agg(['mean', 'count'])
      self.artists_df.loc['unknown'] = [y.mean(), 1]
      self.artists_df.loc[self.artists_df['count'] <= self.MinCnt, 'mean'] = y.mean()
      self.artists_df.loc[self.artists_df['count'] >= self.MaxCnt, 'mean'] = 0
      return self

  def transform(self, X, y=None):
      X['artists'] = np.where(X['artists'].isin(self.artists_df.index), X['artists'], 'unknown')
      X['artists'] = X['artists'].map(self.artists_df['mean'])
      return X

In [ ]:
# Apply AritistsTransformer on train and test seperatly
artists_transformer = ArtistsTransformer(MinCnt=2)
X_train = artists_transformer.fit(X_train, y_train).transform(X_train, y_train)
X_test = artists_transformer.transform(X_test, y_test)

### **Instrumentalness Transformer**

In [ ]:
def instrumentalness_criteria(X):
    X['instrumentalness'] = list(map((lambda x: 1 if x < 0.1 else (3 if x > 0.95 else 2)), X.instrumentalness))

instrumentalness_tranformer = FunctionTransformer(instrumentalness_criteria)
instrumentalness_tranformer.transform(X_train)
instrumentalness_tranformer.transform(X_test)

### **Tempo Transformer**

In [ ]:
class ReplaceZeroTransformer():
    """Eliminates Zero values from tempo columns and replace it 
       with the median or mean of non-zero values as specified.
       defaut is set to 'median'.
    """

    def __init__(self, method='median'):
        self.method = method

    def transform(self, X):
        if self.method == 'median':
            X.loc[X['tempo']==0, 'tempo'] = X.loc[X['tempo']>0, 'tempo'].median()
        elif self.method == 'mean':
            X.loc[X['tempo']==0, 'tempo'] = X.loc[X['tempo']>0, 'tempo'].mean()
        else:
            raise Exception("Method can be 'median' or 'mean' only!")
        return X

In [ ]:
tempo_transformer = ReplaceZeroTransformer()
X_train = tempo_transformer.transform(X_train)
X_test = tempo_transformer.transform(X_test)

### **OneHotEncoder**

We use OneHotEncoder from SKlearn to create dummies

In [ ]:
ohe = OneHotEncoder(categories='auto', drop='first')

# Train
feature_arr = ohe.fit_transform(X_train[['instrumentalness','key']]).toarray()
columns_key = ['key_'+str(i) for i in list(set(X_train['key'].values))[1:]]
instrumentalness_key = ['ins_'+str(i) for i in list(set(X_train['instrumentalness'].values))[1:]]
feature_labels = columns_key + instrumentalness_key
feature_labels = np.concatenate((feature_labels), axis=None)
features = pd.DataFrame(feature_arr, columns = feature_labels, index = X_train.index)
X_train = pd.concat([X_train, features], axis=1).drop(['key','instrumentalness'], axis=1)

# Test
feature_arr = ohe.fit_transform(X_test[['instrumentalness','key']]).toarray()
columns_key = ['key_'+str(i) for i in list(set(X_test['key'].values))[1:]]
instrumentalness_key = ['ins_'+str(i) for i in list(set(X_test['instrumentalness'].values))[1:]]
feature_labels = columns_key + instrumentalness_key
feature_labels = np.concatenate((feature_labels), axis=None)
features = pd.DataFrame(feature_arr, columns = feature_labels, index = X_test.index)
X_test = pd.concat([X_test, features], axis=1).drop(['key','instrumentalness'], axis=1)

### **MinMaxScaler Encoder**

In [ ]:
scaler = MinMaxScaler()
cols = ['artists','duration_ms','loudness','tempo']
X_train[cols] = scaler.fit_transform(X_train[cols])
X_test[cols] = scaler.fit_transform(X_test[cols])

### **Target scaling**

In [ ]:
# Divide the popularity by 100
y_train = y_train / 100
y_test = y_test / 100

### **Verification**

In [ ]:
nl = '\n'
print(f'X_train shape is: {X_train.shape} {nl}y_train shape is: {y_train.shape} {nl}X_test shape is: {X_test.shape} {nl}y_test shape is: {y_test.shape}')

In [ ]:
X_train.head(3)

In [ ]:
X_train.describe().drop(['count','25%', '50%', '75%'])

In [ ]:
y_train.describe().drop(['count','25%', '50%', '75%'])

# **Models**

## **Linear Regression**

### **Option 1: selected features only (corr > 0.2)**

In [ ]:
LR = LinearRegression()
cols = [col for col in X_train.columns if abs(X_train[col].corr(y_train))>0.2]

# Fit the model and
LR.fit(X_train.drop(columns=cols), y_train)

# Train Predicting with the model
y_train_pred = LR.predict(X_train.drop(columns=cols)).clip(0, 1)

# RMSE Train
LR_rmse = np.sqrt(mse(y_train, y_train_pred))
print(f"RMSE Train = {LR_rmse:.5f}")

#Predicting with the model
y_test_pred = LR.predict(X_test.drop(columns=cols)).clip(0, 1)

# RMSE Test
LR_rmse = np.sqrt(mse(y_test, y_test_pred))
print(f"RMSE Test = {LR_rmse:.5f}")

### **Option 2: All features**

In [ ]:
LR = LinearRegression()

# Fit the model and
LR.fit(X_train, y_train)

# Train Predicting with the model
y_train_pred = LR.predict(X_train).clip(0, 1)

# RMSE Train
LR_rmse = np.sqrt(mse(y_train, y_train_pred))
print(f"RMSE Train = {LR_rmse:.6f}")

#Predicting with the model
y_test_pred = LR.predict(X_test).clip(0, 1)

# RMSE Test
LR_rmse = np.sqrt(mse(y_test, y_test_pred))
print(f"RMSE Test = {LR_rmse:.6f}")

In [ ]:
from IPython.display import display, Math

def get_printable_equt(df, inter, coef):
  equtation = '$y = ' + f'{inter:.2f} '
  
  for i, co in enumerate(coef):
    if i == len(coef) - 1:
        equtation = equtation + f'+ {co:.2f} {(df.columns)[i]}$'
    else:
        equtation = equtation + f'+ {co:.2f} {(df.columns)[i]}'

  return display(Math(equtation))

get_printable_equt(X_train, LR.intercept_, LR.coef_)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax = sns.scatterplot(x=y_test, y=y_test_pred)
sns.lineplot(x=y_test, y=y_test, color='red', ax=ax)
ax.set_xlabel('Y_test')
ax.set_ylabel('Y_test_pred')
ax.set_title('y_test vs. y_test_pred', fontsize=14, color='red')
plt.show()

In [ ]:
import statsmodels.api as sm
model = sm.OLS(y_train, X_train).fit()
model.summary()

## **KNN Model**

Due to long running time, we sampled the original data with frac = 0.3.

### **5 to 100 neighbors**

In [ ]:
RMSE1_train, RMSE1_test = [], []
  
for i in range(5,101,5):
  knn = KNeighborsRegressor(n_neighbors=i)
  knn.fit(X_train,y_train)
  y_train_pred = knn.predict(X_train)
  knn_train_rmse = np.sqrt(mse(y_train, y_train_pred))
  RMSE1_train.append(knn_train_rmse.round(3))
  y_test_pred = knn.predict(X_test)
  knn_test_rmse = np.sqrt(mse(y_test, y_test_pred))
  RMSE1_test.append(knn_test_rmse.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(15,4))
x = np.arange(5, 101, 5)
ax = sns.lineplot(x=x, y=RMSE1_train)
sns.lineplot(x=x, y=RMSE1_test, ax=ax)
ax.legend(labels=['RMSE_train', 'RMSE_test'], bbox_to_anchor=(1, 1))
ax.set_xlabel('Neighbors')
ax.set_ylabel('RMSE')
ax.set_xticks(np.arange(5,101,5))
ax.set_title('KNN with 10 <= n-neighbors < 100', c='r', fontdict={'c':'r', 'fontsize':14, 'weight':'bold'})
plt.show()

In [ ]:
gap1 = [RMSE1_test[num]-RMSE1_train[num] for num, i in enumerate(RMSE1_train)]
print(f'RMSE Train: {RMSE1_train[gap1.index(min(gap1))]}, RMSE_test: {RMSE1_test[gap1.index(min(gap1))]}')

### **100 to 200 neighbors**

In [ ]:
RMSE_train, RMSE_test = [], []
  
for i in range(100,201,5):
  knn = KNeighborsRegressor(n_neighbors=i)
  knn.fit(X_train,y_train)
  y_train_pred = knn.predict(X_train)
  knn_train_rmse = np.sqrt(mse(y_train, y_train_pred))
  RMSE_train.append(knn_train_rmse.round(3))
  y_test_pred = knn.predict(X_test)
  knn_test_rmse = np.sqrt(mse(y_test, y_test_pred))
  RMSE_test.append(knn_test_rmse.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(15,4))
x = np.arange(100, 201, 5)
ax = sns.lineplot(x=x, y=RMSE_train)
sns.lineplot(x=x, y=RMSE_test, ax=ax)
# ax.axvline(x=180, ymin=0, ymax=0.3, color='blue')
# ax.axvline(x=139, ymin=0.5, ymax=0.8, color='orange')
ax.legend(labels=['RMSE_train', 'RMSE_test'], bbox_to_anchor=(1, 1))
ax.set_xticks(np.arange(100,201,5))
ax.set_xlabel('K neighbours')
ax.set_ylabel('RMSE')
ax.set_title('KNN with 100 <= K < 201', c='r', fontdict={'c':'r', 'fontsize':14, 'weight':'bold'})
plt.show()

In [ ]:
gap2 = [RMSE_test[num]-RMSE_train[num] for num, i in enumerate(RMSE_train)]
print(f'RMSE Train: {RMSE_train[gap2.index(min(gap2))]}, RMSE_test: {RMSE_test[gap2.index(min(gap2))]}')

## **Decision Tree Model**

### **Option 1: single run**

In [ ]:
# single run
tree = DecisionTreeRegressor(max_leaf_nodes=41, min_samples_split=2000)
tree.fit(X_train, y_train)
y_train_pred = tree.predict(X_train).clip(0, 1)
train_rmse = np.sqrt(mse(y_train, y_train_pred))
y_test_pred = tree.predict(X_test).clip(0, 1)
test_rmse = np.sqrt(mse(y_test, y_test_pred))
print(f'train: {train_rmse}')
print(f'test: {test_rmse}')

### **Option 2: loop**

In [ ]:
RMSE3_train, RMSE3_test = [], []

for i in range(2,200):
  tree = DecisionTreeRegressor(random_state = 15, max_leaf_nodes=i)
  tree.fit(X_train, y_train)
  y_train_pred = tree.predict(X_train).clip(0, 1)
  train_rmse = np.sqrt(mse(y_train, y_train_pred))
  RMSE3_train.append(train_rmse.round(3))
  y_test_pred = tree.predict(X_test).clip(0, 1)
  test_rmse = np.sqrt(mse(y_test, y_test_pred))
  RMSE3_test.append(test_rmse.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(15,4))
x = np.arange(2, 200)
ax = sns.lineplot(x=x, y=RMSE3_train)
sns.lineplot(x=x, y=RMSE3_test, ax=ax)
ax.axvline(x=178, ymin=0, ymax=0.3, color='blue')
# ax.axvline(x=139, ymin=0.5, ymax=0.8, color='orange')
ax.legend(labels=['RMSE_train', 'RMSE_test'], bbox_to_anchor=(1, 1))
ax.set_xlabel('Max Leaf Nodes')
ax.set_ylabel('RMSE')
ax.set_title('DT with 2<= max_leaf_nodes < 200', c='r', fontdict={'c':'r', 'fontsize':14, 'weight':'bold'})
plt.show()

In [ ]:
ax = sns.scatterplot(x=y_train, y=y_train_pred)
ax.plot(y_train, y_train, 'r')
ax.set_ylabel('popularity prediction')
plt.show()

In [ ]:
gap3 = [(RMSE3_train[i]/RMSE3_test[i]) for i, num in enumerate(RMSE3_train)]

print(f'Minimum validiation is: {min(gap3):.3%} in index number {gap3.index(min(gap3))}')
print(f'Maximum validiation is: {max(gap3):.3%} in index number {gap3.index(max(gap3))}')

In [ ]:
print('Feature importances:')
for i, col in enumerate(X_train.columns):
  print(f'{col:12}: {tree.feature_importances_[i]:.3f}')

In [ ]:
dot_data = StringIO()
export_graphviz(tree, out_file=dot_data,feature_names=X_train.columns)
graph = pydotplus.graph_from_dot_data(dot_data.getvalue())
Image(graph.create_png())

# **Conclusion**

In this notebook we explored the data, dropped unnecessary features, and tested several regression models: Linear Regression, KNN & DT.

We also tested the data with different test size, in order to check the influence on the results.

Eventually, the best model was a Decision Tree run with test size = 0.2, max_leaf_nodes = 178, mincnt = 2.

This gave a RMSE train of 0.102 and RMSE test of 0.112.

A pdf version of the summary presentation is available here: [GitHub](https://github.com/pelanat1207/Spotify-Project)